# Análise das Condições de Admissão 
(Empresa Fictícia)

In [ ]:
import pandas as pd
from scipy import stats

import plotly.express as px

## Pré-processamento

### Carregar base de dados

In [ ]:
df = pd.read_csv('historico-de-admissoes.csv')
display(df.sample(5))
print(df.info())

### Ajustar colunas

#### Salário para valor numérico

In [ ]:
df['SALÁRIO'] = (
    df['SALÁRIO']
    .astype(str)
    .str.replace('R$', '', regex=False)
    .str.replace('\xa0', '', regex=False)
    .str.replace(' ', '', regex=False)
    .str.replace('.', '', regex=False)
    .str.replace(',', '.', regex=False)
    .astype(float)
)

#### Textos para datas

In [ ]:
df['DATA DE ADMISSÃO'] = pd.to_datetime(df['DATA DE ADMISSÃO'], format='%d/%m/%Y', errors='coerce')
df['DATA DE NASCIMENTO'] = pd.to_datetime(df['DATA DE NASCIMENTO'], format='%d/%m/%Y', errors='coerce')

#### Formatar valores de texto

In [ ]:
cols_categoria = ['CARGO', 'DESCRIÇÃO ESCOLARIDADE', 'SEXO', 'PCD', 'DIRETORIA', 'SUPERINTENDÊNCIA']
for col in cols_categoria:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip()

### Base ajustada

In [ ]:
display(df.sample(5))
print(df.info())

## Funções Auxiliares

In [ ]:
def plot_salary_boxplot(df:pd.DataFrame, x:str, x_label:str):
    figure = px.box(
        df, x=x, y='SALÁRIO', color=x, points='all',
        title=f'Distribuição Salarial por {x_label} na Admissão',
        labels={x: x_label, 'SALÁRIO': 'Salário Admissional (R$)'}
    )
    figure.show()

In [ ]:
def check_salary_assumptions(df:pd.DataFrame, x:str, alpha:float = 0.05, details=False):
    groups = df[x].dropna().unique()
    num_groups = groups.__len__()


    # Teste de Normalidade
    normal_classes = []
    not_normal_classes = []
    less_than_tree_instances_classes = []
    for g in groups:
        group_data = df[df[x] == g]["SALÁRIO"]
        if len(group_data) >= 3:
            stat_w, p_val_w = stats.shapiro(group_data)
            if p_val_w > alpha:
                normal_classes.append(g)
            else:
                not_normal_classes.append(g)
        else:
            less_than_tree_instances_classes.append(g)

    is_normal = (normal_classes.__len__() == num_groups)


    # Teste de Homocedasticidade
    data_per_group = [df[df[x] == g]["SALÁRIO"] for g in groups]
    stat_l, p_val_l = stats.levene(*data_per_group)

    is_homocedastic = p_val_l > alpha


    print(f"Avaliação de pré-requisitos ({x}):")

    if details:
        print("\nTeste de Normalidade (Shapiro-Wilk)")
        print(f" - Classes Normais {normal_classes}")
        print(f" - Classes Não Normais {not_normal_classes}")
        print(f" - Classes de Instancias Insuficientes {less_than_tree_instances_classes}")

        print("\nTeste de Homogeneidade de Variâncias (Levene)")
        print(f" - Classes {'são' if is_homocedastic else 'não são'} homocedasticas")

    print(f" - Normalidade: {'Sim' if is_normal else 'Não'}")
    print(f" - Homocedasticidade: {'Sim' if is_homocedastic else 'Não'}")

    print("\nTeste(s) recomendados:")
    if is_normal and is_homocedastic:
        if num_groups > 2:
            print(" - ANOVA One-Way + Tukey HSD")
        else:
            print(" - Teste t de Student")
    else:
        if num_groups > 2:
            print(" - Kruskal-Wallis + Teste de Dunn")
        else:
            print(" - Teste Mann-Whitney U")